In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
from solver import *

## 2D Viscous Burgers' Equation (Simplified Navier-Stokes)

In [ ]:
# Symbolic variables
t, x, y, xi, eta = symbols('t x y xi eta', real=True)
u = Function('u')

# 2D Burgers' equation: ∂u/∂t + u∂u/∂x + u∂u/∂y = ν(∂²u/∂x² + ∂²u/∂y²)
# Using psiOp for the viscous term (works with Dirichlet BC)
nu = 0.01  # Viscosity
equation = Eq(
    diff(u(t, x, y), t), 
    -u(t, x, y)*diff(u(t, x, y), x) - u(t, x, y)*diff(u(t, x, y), y)
    - psiOp(xi**2 + eta**2, nu * u(t, x, y))
)

# Solver creation
solver = PDESolver(equation)

# Parameters
Lx, Ly = 2 * np.pi, 2 * np.pi
Nx, Ny = 128, 128
Lt = 1.0
Nt = 200

# Initial condition: Smooth vortex-like pattern (vanishes at boundaries)
def initial_condition(x, y):
    return 0.5 * np.sin(x) * np.sin(y) * np.exp(-0.5 * (x**2 + y**2))

# Setup with Dirichlet BC (no-slip walls)
solver.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='dirichlet',
    initial_condition=initial_condition
)

# Solve
solver.solve()

# Visualize evolution
ani = solver.animate(component='real', mode='surface', overlay='contour')
HTML(ani.to_jshtml())

## 2D Kuramoto-Sivashinsky Equation (Pattern Formation)

In [ ]:
# Symbolic variables
t, x, y, xi, eta = symbols('t x y xi eta', real=True)
u = Function('u')

# Kuramoto-Sivashinsky: ∂u/∂t = -∇²u - ∇⁴u - ½|∇u|²
# Models flame fronts, fluid interfaces, spatiotemporal chaos
equation = Eq(
    diff(u(t, x, y), t),
    -psiOp(xi**2 + eta**2 + (xi**2 + eta**2)**2, u(t, x, y))  # Anti-diffusion (instability) + Hyper-diffusion (stabilization)
#    -0.5 * (diff(u(t, x, y), x)**2 + diff(u(t, x, y), y)**2)  # Nonlinearity
)

# Solver creation
solver = PDESolver(equation)

# Parameters
Lx, Ly = 2 * np.pi, 2 * np.pi
Nx, Ny = 256, 256  # Higher resolution for pattern formation
Lt = 5.0
Nt = 500

# Initial condition: Small random perturbation
np.random.seed(42)
def initial_condition(x, y):
    return 0.1 * np.sin(x) * np.sin(y) + 0.01 * np.random.randn(*x.shape)

# Setup
solver.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='dirichlet',
    initial_condition=initial_condition,
    n_frames=100
)

# Solve
solver.solve()

# Visualize evolution
ani = solver.animate(component='real', mode='surface', overlay='contour')
HTML(ani.to_jshtml())

In [ ]:
# Symbolic variables
t, x, y, xi, eta = symbols('t x y xi eta', real=True)
omega = Function('omega')  # Vorticity

# 2D vorticity equation: ∂ω/∂t + J(ψ,ω) = ν∇²ω
# where ψ is streamfunction (∇²ψ = -ω) and J is Jacobian
# Simplified: ∂ω/∂t = -u·∇ω + ν∇²ω
nu = 0.001  # Kinematic viscosity

equation = Eq(
    diff(omega(t, x, y), t),
    -psiOp(xi**2 + eta**2, nu * omega(t, x, y))  # Viscous diffusion
)

# Note: Advection term handled separately in nonlinear terms
# For full implementation, add: -diff(psi, y)*diff(omega, x) + diff(psi, x)*diff(omega, y)

# Solver creation
solver = PDESolver(equation)

# Parameters
Lx, Ly = 2 * np.pi, 2 * np.pi
Nx, Ny = 256, 256
Lt = 2.0
Nt = 300

# Initial condition: Taylor-Green vortex (analytic solution for Navier-Stokes)
def initial_condition(x, y):
    return np.sin(x) * np.sin(y) * np.cos(x) * np.cos(y)


# Setup
solver.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='dirichlet',
    initial_condition=initial_condition
)

# Solve
solver.solve()


# Visualize evolution
ani = solver.animate(component='real', mode='surface', overlay='contour')
HTML(ani.to_jshtml())

In [ ]:
# Symbolic variables
t, x, y, xi, eta = symbols('t x y xi eta', real=True)
u = Function('u')

# Fractional advection-diffusion: ∂u/∂t + v·∇u = -D(-Δ)^α u
# Models anomalous transport in porous media, turbulent flows
alpha = 0.75  # Fractional order (0.5 < α < 1 for super-diffusion)
D = 0.1       # Diffusion coefficient
vx, vy = 0.5, 0.3  # Advection velocity

equation = Eq(
    diff(u(t, x, y), t),
#    -vx * diff(u(t, x, y), x) - vy * diff(u(t, x, y), y)
    - psiOp(((xi**2 + eta**2)**alpha -vx * xi -vy * eta) / D, u(t, x, y))
)

# Solver creation
solver = PDESolver(equation)

# Parameters
Lx, Ly = 2 * np.pi, 2 * np.pi
Nx, Ny = 256, 256
Lt = 3.0
Nt = 300

# Initial condition: Localized pollutant plume
def initial_condition(x, y):
    return np.exp(-5 * ((x - 0.5)**2 + (y - 0.5)**2)) * np.sin(x) * np.sin(y)

# Setup
solver.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='dirichlet',
    initial_condition=initial_condition
)

# Solve
solver.solve()

# Visualize evolution
ani = solver.animate(component='real', mode='surface', overlay='front')
HTML(ani.to_jshtml())